# DDPG — Continuous control with deep reinforcement learning (2015)

_Papers · Reinforcement Learning_

**Make deep Q-learning work for continuous actions by pairing a deterministic actor with a Q-critic, trained off-policy with a replay buffer and slowly-tracking target networks.**

---

This notebook is a practice scaffold for the **DDPG — Continuous control with deep reinforcement learning (2015)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — gymnasium + PyTorch (Colab)

In [ ]:
# In Colab first run:  !pip install gymnasium
# (torch is preinstalled in Colab; gymnasium is not.)
import copy, random
import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym

torch.manual_seed(0); np.random.seed(0); random.seed(0)

# --- 0. Sanity-check the lesson's worked numbers. ---
GAMMA, TAU = 0.99, 0.01
r, Qp = -1.0, -5.0                     # reward, target-critic score of next action
y = r + GAMMA * Qp                     # Bellman target = -1.0 + 0.99*(-5.0) = -5.95
err = (y - (-5.50)) ** 2               # critic squared error vs live Q = -5.50 -> 0.2025
pg  = (2.0) * (0.5)                     # Eq. 6 term: dQ/da * dmu/dtheta = 1.0
soft = TAU * 3.00 + (1 - TAU) * 2.00   # soft update of theta'=2.00 toward theta=3.00 -> 2.01
print("worked example:  y =", round(y, 4), " critic_err =", round(err, 4),
      " policy_grad_term =", round(pg, 4), " soft_updated_target =", round(soft, 4))
# worked example:  y = -5.95  critic_err = 0.2025  policy_grad_term = 1.0  soft_updated_target = 2.01


# --- 1. Actor (deterministic policy) and Critic (Q-network). ---
class Actor(nn.Module):
    def __init__(self, obs_dim, act_dim, act_limit, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, act_dim), nn.Tanh())  # tanh -> [-1, 1]
        self.act_limit = act_limit
    def forward(self, s):
        return self.act_limit * self.net(s)            # squash + scale to action range

class Critic(nn.Module):                               # Q(s, a): state and action concatenated
    def __init__(self, obs_dim, act_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim + act_dim, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, 1))
    def forward(self, s, a):
        return self.net(torch.cat([s, a], dim=-1)).squeeze(-1)


# --- 2. Replay buffer (off-policy) and Ornstein-Uhlenbeck exploration noise. ---
class ReplayBuffer:
    def __init__(self, cap=100_000): self.buf = []; self.cap = cap
    def add(self, *t):
        self.buf.append(t)
        if len(self.buf) > self.cap: self.buf.pop(0)
    def sample(self, n):
        batch = random.sample(self.buf, n)
        s, a, rw, s2, d = map(np.array, zip(*batch))
        f = lambda x: torch.as_tensor(x, dtype=torch.float32)
        return f(s), f(a), f(rw), f(s2), f(d)
    def __len__(self): return len(self.buf)

class OUNoise:                                          # temporally-correlated exploration
    def __init__(self, dim, theta=0.15, sigma=0.2):
        self.theta, self.sigma, self.dim = theta, sigma, dim; self.reset()
    def reset(self): self.x = np.zeros(self.dim)
    def __call__(self):
        self.x = self.x - self.theta * self.x + self.sigma * np.random.randn(self.dim)
        return self.x


# --- 3. Soft target update: theta' <- tau*theta + (1-tau)*theta'  (the stability key). ---
def soft_update(target, source, tau):
    for pt, p in zip(target.parameters(), source.parameters()):
        pt.data.mul_(1.0 - tau).add_(tau * p.data)


# --- 4. Train DDPG on Pendulum; PRINT the average return rising toward 0. ---
def train(use_targets=True, episodes=60, batch=128, gamma=0.99, tau=0.005,
          start_steps=1000):
    torch.manual_seed(0); np.random.seed(0); random.seed(0)
    env = gym.make("Pendulum-v1")
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.shape[0]
    act_limit = float(env.action_space.high[0])

    actor  = Actor(obs_dim, act_dim, act_limit)
    critic = Critic(obs_dim, act_dim)
    actor_t  = copy.deepcopy(actor)                    # target actor  mu'
    critic_t = copy.deepcopy(critic)                   # target critic Q'
    a_opt = torch.optim.Adam(actor.parameters(),  lr=1e-3)
    c_opt = torch.optim.Adam(critic.parameters(), lr=1e-3)
    buf = ReplayBuffer(); noise = OUNoise(act_dim)
    total_steps = 0; history = []

    for ep in range(episodes):
        s, _ = env.reset(seed=ep); noise.reset(); ep_ret = 0.0; done = False
        while not done:
            if total_steps < start_steps:              # warm up with random actions
                a = env.action_space.sample()
            else:
                with torch.no_grad():
                    a = actor(torch.as_tensor(s, dtype=torch.float32)).numpy()
                a = np.clip(a + noise(), -act_limit, act_limit)   # mu(s) + OU noise
            s2, rw, term, trunc, _ = env.step(a)
            done = term or trunc
            buf.add(s, a, rw, s2, float(done)); s = s2
            ep_ret += rw; total_steps += 1

            if len(buf) >= batch:                      # one gradient update per step
                S, A, R, S2, D = buf.sample(batch)
                # --- critic update: regress Q(s,a) toward the Bellman target y ---
                with torch.no_grad():
                    if use_targets:                    # y from the slow TARGET networks
                        a2 = actor_t(S2)
                        q2 = critic_t(S2, a2)
                    else:                              # ABLATION: bootstrap off LIVE nets
                        a2 = actor(S2)
                        q2 = critic(S2, a2)
                    y = R + gamma * (1.0 - D) * q2
                q = critic(S, A)
                critic_loss = ((q - y) ** 2).mean()    # mean-squared Bellman error
                c_opt.zero_grad(); critic_loss.backward(); c_opt.step()
                # --- actor update: deterministic policy gradient, Eq. 6 ---
                actor_loss = -critic(S, actor(S)).mean()   # maximize Q at mu(s)
                a_opt.zero_grad(); actor_loss.backward(); a_opt.step()
                # --- soft target update (skipped in the ablation) ---
                if use_targets:
                    soft_update(actor_t,  actor,  tau)
                    soft_update(critic_t, critic, tau)
        history.append(ep_ret)
        if (ep + 1) % 10 == 0:
            print(f"  episode {ep+1:3d}  avg return (last 10): "
                  f"{np.mean(history[-10:]):8.1f}")
    env.close()
    return history

print("DDPG with soft-updated target networks:")
with_t = train(use_targets=True)
print("\nABLATION -- no target networks (bootstrap off live nets, same everything else):")
no_t = train(use_targets=False)
print("\nWith-targets last-10 avg:", round(float(np.mean(with_t[-10:])), 1))
print("No-targets   last-10 avg:", round(float(np.mean(no_t[-10:])), 1))
# With soft-updated target networks the average return climbs toward 0; the no-target
# ablation learns erratically and plateaus lower. (Numbers vary by hardware/seed; our
# small run, not the paper's reported results.)

## Visualize the data & results

_Does DDPG's average return on Pendulum rise toward 0, and do the soft-updated target networks matter? We train DDPG with target networks and an ablation that drops them (bootstrapping off the live actor/critic, everything else identical) and plot the per-episode return._

In [ ]:
# Sketch of how the two curves above are produced (full loop in the CODE cell).
# Train DDPG with target networks and the no-target ablation on Pendulum-v1 for the same
# episodes with identical actor/critic / buffer / OU noise / lr / seed, recording the
# per-episode return.
#
#   with_t = train(use_targets=True)    # green: climbs from ~-1480 toward ~-150
#   no_t   = train(use_targets=False)   # red:   erratic, stuck near ~-1100
#
# The points plotted are the per-episode return.
# With targets -> steady climb toward 0 (Pendulum's reward ceiling).
# No targets   -> the critic regresses toward a target that moves as fast as it does;
#                 the bootstrap chases its own tail and never settles.
# (Numbers are from our own run; the paper reports solving >20 physics tasks with one
#  set of hyper-parameters, not these Pendulum values.)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The worked target + soft update. With $\gamma = 0.99$ and $\tau = 0.01$: a transition gave
            reward $r = -1.0$; the target actor picks $\mu'(s') = 0.30$ and the target critic scores it
            $Q'(s', 0.30) = -5.0$; the live critic currently predicts $Q(s,a) = -5.50$. Compute the Bellman target
            $y$, the critic's squared error, and &mdash; given a target weight $\theta' = 2.00$ and live weight
            $\theta = 3.00$ &mdash; the soft-updated $\theta'$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Bellman target: $y = r + \gamma\,Q'(s',\mu'(s')) = -1.0 + 0.99\times(-5.0)$. — _The critic's regression target is reward now plus discounted next-state value, where the next action and its value both come from the TARGET networks._
- So $y = -1.0 - 4.95 = -5.95$. — _That is the constant the critic is regressed toward this step (computed under no_grad)._
- Squared error: $(y - Q(s,a))^2 = (-5.95 - (-5.50))^2 = (-0.45)^2 = 0.2025$. — _Minimizing this mean-squared error is the critic's loss; the gradient step shrinks the $-0.45$ gap._
- Soft update: $\theta' \leftarrow 0.01\times 3.00 + 0.99\times 2.00 = 0.03 + 1.98 = 2.01$. — _The target moves a fraction $\tau = 1\%$ toward the live weight &mdash; a slow exponential moving average._

**Answer:** $y = -5.95$, squared error $= 0.2025$, and the soft-updated target weight $= 2.01$ (it crept
                 just $0.01$ toward the live $3.00$). The notebook recomputes all three:
                 $-1.0 + 0.99\cdot(-5.0) = -5.95$, $(-0.45)^2 = 0.2025$, $0.01\cdot 3 + 0.99\cdot 2 = 2.01$.

</details>

**Problem 2.** The ablation. Your DDPG agent's average return on Pendulum rises toward $0$. Now remove the
            target networks: compute the Bellman target from the live critic and actor instead
            ($y = r + \gamma\,Q(s', \mu(s'))$), keeping everything else (networks, replay buffer, OU noise,
            learning rates, seed) identical, and retrain. What happens to the return curve, and what does that
            demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change only the target: use the live $Q$ and $\mu$ to build $y$ (drop $Q'$, $\mu'$, and the soft update); keep depth, optimizer, buffer, noise, and seed fixed. — _An honest ablation changes exactly one thing &mdash; the target networks &mdash; so any difference is attributable to them._
- Retrain and watch the return: the no-target run learns erratically and plateaus far below the soft-target run, often oscillating instead of converging. — _Without a slow-moving target, the critic regresses toward a target that shifts as fast as it does; the bootstrap chases its own tail and the value estimates wander._
- Conclude the soft-updated target networks, not the actor or buffer, are what make the off-policy bootstrap stable. — _Both runs share architecture, data, and noise; only the target-network run is stable, isolating the soft update as the cause._

**Answer:** The no-target agent destabilizes &mdash; its return is erratic and stays well below the
                 soft-target agent, which climbs steadily toward $0$. Since the only difference is whether the
                 Bellman target is built from slow-moving target networks, this isolates the soft-updated target
                 networks as the source of stability: they turn an otherwise-divergent bootstrap into a stationary
                 regression target. The CODEVIZ panel shows this contrast.

</details>

**Problem 3.** Suppose at some state the critic's action-slope is $\nabla_a Q = -3.0$ (raising the action lowers
            value) and the actor's parameter-slope is $\nabla_{\theta^\mu}\mu = +0.5$. Using Eq. 6, what is the
            policy-gradient contribution for this sample, and which way will the actor's weight move when we
            minimize $-Q(s,\mu(s))$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Eq. 6 product: $\nabla_a Q \cdot \nabla_{\theta^\mu}\mu = (-3.0)\times(0.5) = -1.5$. — _This is $\nabla_{\theta^\mu} J$ for the sample &mdash; the ascent direction on $Q$._
- We ascend $J$ (descend $-Q$), so the weight moves in the direction of $\nabla_{\theta^\mu} J = -1.5$, i.e. the weight DECREASES. — _Gradient ascent on $J$ adds a positive multiple of $\nabla_{\theta^\mu} J$; here that term is negative, so the weight goes down._
- Check the logic: $\nabla_a Q \lt 0$ means a smaller action gives more value, and $\nabla_{\theta^\mu}\mu \gt 0$ means decreasing the weight decreases the action &mdash; consistent. — _The actor is pushed exactly so its output action moves toward higher critic value._

**Answer:** The policy-gradient contribution is $-3.0\times 0.5 = -1.5$. Ascending the actor objective $J$
                 (equivalently minimizing $-Q$) moves the weight in the negative direction &mdash; the weight
                 decreases, which lowers the action, which (since $\nabla_a Q \lt 0$) raises $Q$. Eq. 6
                 simply routes "make the action more valuable" back into a weight update, with no
                 argmax anywhere.

</details>